# Global causal-claim alignment

Compare the directional semantic coverage of extracted causal claims in policy documents and public feedback. No topic model, topic grouping, or geographic partitions are used.

**Interpretation:** A high cosine score reflects textual similarity; a missing match indicates a potential representational gap, not a proven causal discrepancy.


## Environment and embeddings
Reuses the original multilingual embedding configuration (with the original TF–IDF/LSA fallback).


In [1]:
# Compare extracted causal statements across two empirical corpora.
# Every extracted claim is compared across the pooled empirical corpora; no topic partition is used.
# Sentence-level matching supplies evidence of textual coverage, not causal effects.
# See README.md for method scope and interpretation notes.

import html
import io
import os

# Keep sparse linear-algebra backends deterministic and avoid oversubscription.
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
import re
from pathlib import Path
import shutil

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import Image as NotebookImage, display

from scipy.spatial.distance import jensenshannon
from scipy.sparse import hstack
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 220)

RANDOM_STATE = 42
MIN_COUNTRY_ITEMS = 5
SENTENCE_EMBEDDING_MODEL = os.environ.get(
    "CAUSAL_NLP_EMBEDDING_MODEL",
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
)
USE_SENTENCE_TRANSFORMERS = os.environ.get(
    "CAUSAL_NLP_USE_SENTENCE_TRANSFORMERS", "1"
).strip().lower() not in {"0", "false", "no"}


def find_project_root() -> Path:
    """Locate the project using its combined source-segment dataset."""
    candidates = []
    configured = os.environ.get("CAUSAL_NLP_PROJECT_ROOT")
    if configured:
        candidates.append(Path(configured).expanduser().resolve())
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    for candidate in candidates:
        if (candidate / "data" / "policy_feedback_segments.csv").is_file():
            return candidate
    raise FileNotFoundError("Run from the extracted project or set CAUSAL_NLP_PROJECT_ROOT.")

PROJECT_ROOT = find_project_root()
CSVS_CHUNKED = PROJECT_ROOT / "data"
CHUNKS_ALL = CSVS_CHUNKED / "policy_feedback_segments.csv"
CAUSAL_NLP_ROOT = Path(
    os.environ.get(
        "CAUSAL_NLP_OUTPUT_ROOT",
        PROJECT_ROOT / "causal_semantics",
    )
).expanduser().resolve()
METHOD_ROOT = CAUSAL_NLP_ROOT



def infer_sentiment_country(row: pd.Series) -> str:
    """Derive only clearly supported sentiment country groups."""
    source = " ".join(
        str(row.get(column, ""))
        for column in ["filename", "relative_source_file", "source_file", "doc_id"]
    ).lower()
    if any(token in source for token in ["ireland", "irish times", "qqi_", "qqi "]):
        return "ireland"
    if any(token in source for token in ["france", "français", "francais", "french", "ifop", "labo"]):
        return "france"
    return "other"


def build_context_windows(chunks: pd.DataFrame) -> pd.DataFrame:
    """Add neighbouring original chunks without changing the analytical unit."""
    ordered = chunks.sort_values(["doc_id", "chunk_index"]).copy()
    ordered["previous_chunk_text"] = ordered.groupby("doc_id")["chunk_text"].shift(1)
    ordered["next_chunk_text"] = ordered.groupby("doc_id")["chunk_text"].shift(-1)
    ordered["context_window"] = (
        ordered["heading_context"].fillna("").astype(str)
        + "\n\nPREVIOUS:\n"
        + ordered["previous_chunk_text"].fillna("").astype(str)
        + "\n\nCURRENT:\n"
        + ordered["chunk_text"].fillna("").astype(str)
        + "\n\nNEXT:\n"
        + ordered["next_chunk_text"].fillna("").astype(str)
    ).str.slice(0, 7000)
    return ordered





# Shared causal-text cleaning layer
import sys

_SHARED_CANDIDATES = [
    Path(os.environ.get("CAUSAL_NLP_SHARED_MODULE_DIR", "")).expanduser(),
    PROJECT_ROOT / "causal_semantics",
    Path.cwd() / "shared",
    Path.cwd(),
]
SHARED_MODULE_DIR = next(
    (
        candidate.resolve()
        for candidate in _SHARED_CANDIDATES
        if str(candidate) and (candidate / "causal_text_cleaning.py").exists()
    ),
    None,
)
if SHARED_MODULE_DIR is None:
    raise FileNotFoundError(
        "causal_text_cleaning.py was not found. Place it in "
        "causal_semantics or set CAUSAL_NLP_SHARED_MODULE_DIR."
    )
if str(SHARED_MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(SHARED_MODULE_DIR))

from causal_text_cleaning import (
    CAUSAL_ARTIFACT_PHRASES,
    CAUSAL_ARTIFACT_TOKENS,
    CAUSAL_EMBEDDING_STOPWORDS,
    CLEANING_VERSION,
    build_context_windows,
    contains_contact_or_link,
    content_token_ratio,
    infer_sentiment_country,
    is_source_residue,
    load_or_build_clean_sentence_inventory,
    normalize_source_text,
    sentence_units,
)

# Compatibility names used by the method-specific extraction code.
normalize_for_noise_check = normalize_source_text
split_sentences = sentence_units


def fit_text_embeddings(texts: list[str], max_components: int = 60):
    """Use multilingual Sentence-BERT, otherwise multilingual word+character LSA."""
    if USE_SENTENCE_TRANSFORMERS:
        try:
            from sentence_transformers import SentenceTransformer

            model = SentenceTransformer(
                SENTENCE_EMBEDDING_MODEL,
                local_files_only=True,
            )
            embeddings = model.encode(
                texts,
                normalize_embeddings=True,
                show_progress_bar=False,
            )
            return np.asarray(embeddings), {
                "method": f"sentence_transformer:{SENTENCE_EMBEDDING_MODEL}",
                "sentence_model": model,
                "word_vectorizer": None,
                "char_vectorizer": None,
                "svd": None,
            }
        except Exception as error:
            print(
                "Multilingual Sentence-BERT unavailable; "
                "using multilingual word+character TF-IDF plus LSA:",
                type(error).__name__,
            )

    word_vectorizer = TfidfVectorizer(
        analyzer="word",
        stop_words=sorted(CAUSAL_EMBEDDING_STOPWORDS),
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        sublinear_tf=True,
        max_features=6000,
    )
    char_vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 4),
        min_df=3,
        sublinear_tf=True,
        max_features=6000,
    )
    word_matrix = word_vectorizer.fit_transform(texts)
    char_matrix = char_vectorizer.fit_transform(texts)
    matrix = hstack([word_matrix, char_matrix], format="csr")

    n_components = min(
        max_components,
        matrix.shape[0] - 1,
        matrix.shape[1] - 1,
    )
    if n_components < 2:
        return normalize(matrix), {
            "method": "multilingual_word_char_tfidf_sparse",
            "sentence_model": None,
            "word_vectorizer": word_vectorizer,
            "char_vectorizer": char_vectorizer,
            "svd": None,
        }

    svd = TruncatedSVD(
        n_components=n_components,
        random_state=RANDOM_STATE,
        n_iter=3,
    )
    embeddings = normalize(svd.fit_transform(matrix))
    return embeddings, {
        "method": f"multilingual_word_char_lsa:{n_components}_components",
        "sentence_model": None,
        "word_vectorizer": word_vectorizer,
        "char_vectorizer": char_vectorizer,
        "svd": svd,
    }

def transform_text_embeddings(texts: list[str], backend: dict) -> np.ndarray:
    if backend["sentence_model"] is not None:
        return np.asarray(
            backend["sentence_model"].encode(
                texts,
                normalize_embeddings=True,
                show_progress_bar=False,
            )
        )

    word_matrix = backend["word_vectorizer"].transform(texts)
    char_matrix = backend["char_vectorizer"].transform(texts)
    matrix = hstack([word_matrix, char_matrix], format="csr")
    if backend["svd"] is None:
        return normalize(matrix)
    return normalize(backend["svd"].transform(matrix))

def normalise_minmax(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce").fillna(0.0)
    minimum = float(values.min())
    maximum = float(values.max())
    if maximum <= minimum:
        return pd.Series(np.zeros(len(values)), index=series.index)
    return (values - minimum) / (maximum - minimum)

def save_verified_png(fig, output_path: Path, dpi: int = 120) -> None:
    """Write a standard RGB PNG directly to the final path and verify it."""
    output_path = Path(output_path).expanduser().resolve()
    output_path.parent.mkdir(parents=True, exist_ok=True)

    # Render the matplotlib figure fully in memory first. This avoids temporary
    # files and atomic renames, which can be unreliable on some synced or
    # virtual filesystems.
    raw_buffer = io.BytesIO()
    rgb_buffer = io.BytesIO()

    try:
        fig.savefig(
            raw_buffer,
            format="png",
            dpi=dpi,
            bbox_inches="tight",
            facecolor="white",
            transparent=False,
        )
        plt.close(fig)

        raw_buffer.seek(0)
        with Image.open(raw_buffer) as image:
            image.load()
            rgb_image = image.convert("RGB")
            rgb_image.save(
                rgb_buffer,
                format="PNG",
                optimize=False,
                compress_level=6,
            )

        png_bytes = rgb_buffer.getvalue()
        if not png_bytes.startswith(b"\x89PNG\r\n\x1a\n"):
            raise OSError("Rendered bytes do not contain a valid PNG signature.")
        if len(png_bytes) <= 100:
            raise OSError("Rendered PNG is unexpectedly small.")

        # Remove any stale or incomplete file before writing the final bytes.
        output_path.unlink(missing_ok=True)
        output_path.write_bytes(png_bytes)

        # Reopen the exact final path that will be used by the notebook and the
        # file browser.
        with Image.open(output_path) as image:
            image.load()
            if image.format != "PNG":
                raise OSError(f"Unexpected final image format: {image.format}")
            if image.mode != "RGB":
                raise OSError(f"Unexpected final image mode: {image.mode}")
            width, height = image.size
            if width <= 0 or height <= 0:
                raise OSError("Final PNG has invalid dimensions.")

        print(
            "Saved verified PNG:",
            output_path,
            "mode=RGB",
            f"size=({width}, {height})",
            f"bytes={output_path.stat().st_size:,}",
        )
    finally:
        plt.close(fig)
        raw_buffer.close()
        rgb_buffer.close()

# Input labels are mapped to the internal analytical schema at the CSV boundary.
SOURCE_SCHEMA = {'record_ref': 'doc_id', 'source_label': 'filename', 'source_path': 'source_file', 'relative_path': 'relative_source_file', 'collection': 'corpus', 'origin_type': 'source_type', 'augmentation_type': 'synthetic_type', 'geography': 'country', 'document_characters': 'char_count', 'document_words': 'word_count', 'segment_key': 'chunk_id', 'segment_index': 'chunk_index', 'section_context': 'heading_context', 'segment_content': 'chunk_text', 'segment_characters': 'chunk_char_count', 'segment_words': 'chunk_word_count'}


## Shared sentence evidence
Load the cleaned source sentences directly and attach their original segment contexts.


In [2]:
# Create output folders and reuse the prepared sentence inventory.
OUTPUT_DIR = METHOD_ROOT / "output" / "semantic_coverage"
IMG_DIR = METHOD_ROOT / "figures" / "semantic_coverage"
SHARED_OUTPUT_DIR = METHOD_ROOT / "output" / "shared"
for folder in (OUTPUT_DIR, IMG_DIR, SHARED_OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

CLEAN_SENTENCE_INVENTORY = SHARED_OUTPUT_DIR / "sentence_evidence_cache.csv"
CLEAN_SENTENCE_METADATA = SHARED_OUTPUT_DIR / "sentence_evidence_metadata.json"
CLEAN_SENTENCE_SUMMARY = SHARED_OUTPUT_DIR / "sentence_evidence_summary.csv"
CLEAN_SENTENCE_VALIDATION = SHARED_OUTPUT_DIR / "sentence_evidence_validation.csv"

clean_sentence_inventory, shared_cleaning_metadata = load_or_build_clean_sentence_inventory(
    CHUNKS_ALL, CLEAN_SENTENCE_INVENTORY, CLEAN_SENTENCE_METADATA,
    CLEAN_SENTENCE_SUMMARY, CLEAN_SENTENCE_VALIDATION,
)
raw_chunks_all = build_context_windows(pd.read_csv(CHUNKS_ALL).rename(columns=SOURCE_SCHEMA))
context_columns = raw_chunks_all[["chunk_id", "chunk_text", "context_window"]].drop_duplicates("chunk_id")

def select_sentences(corpus: str, source_type: str) -> pd.DataFrame:
    # Join sentence-level evidence directly to its source segment; do not filter by topics.
    selected = clean_sentence_inventory[
        clean_sentence_inventory["corpus"].eq(corpus)
        & clean_sentence_inventory["source_type"].eq(source_type)
    ].copy().merge(context_columns, on="chunk_id", how="left", validate="many_to_one")
    selected["original_chunk_text"] = selected["chunk_text"]
    selected["chunk_text"] = selected["clean_sentence"]
    selected["analysis_country"] = "global"
    return selected

policy_chunks = select_sentences("policy", "original")
sentiment_chunks = select_sentences("sentiment", "original")
synthetic_chunks = select_sentences("sentiment", "synthetic")

inventory_coverage = pd.DataFrame([
    {"source_set": label, "sentences": len(frame), "source_segments": frame["chunk_id"].nunique(),
     "documents": frame["doc_id"].nunique()}
    for label, frame in (("policy", policy_chunks), ("commentary", sentiment_chunks),
                         ("synthetic_probe", synthetic_chunks))
])
inventory_coverage.to_csv(OUTPUT_DIR / "global_sentence_inventory_summary.csv", index=False)
print(inventory_coverage.to_string(index=False))


     source_set  sentences  source_segments  documents
         policy      15281             1930         56
     commentary       4182              522         17
synthetic_probe       2128              212         53


## Causal-language extraction
Preserve the cue patterns, span filters, and deduplication logic from the original method.


In [3]:
# --- Analysis stage ---------------------------------------------------------

MAX_SENTENCE_WORDS = 110
MIN_SIDE_WORDS = 2
MAX_SIDE_WORDS = 65


def cause_first_pattern(cues: str) -> str:
    """Build a boundary-safe cause → cue → effect expression."""
    return (
        rf"(?P<cause>.+?)\s+"
        rf"(?P<cue>(?<!\w)(?:{cues})(?!\w))\s+"
        rf"(?P<effect>.+)"
    )


def reversed_pattern(cues: str) -> str:
    """Build a boundary-safe effect → cue → cause expression."""
    return (
        rf"(?P<effect>.+?)\s+"
        rf"(?P<cue>(?<!\w)(?:{cues})(?!\w))\s+"
        rf"(?P<cause>.+)"
    )


# Patterns are ordered from specific multi-word cues to broader predicates.
CAUSE_FIRST_PATTERNS = [
    (
        cause_first_pattern(
            r"may lead to|can lead to|could lead to|is likely to lead to|"
            r"lead(?:s|ing)? to|gave rise to|gives? rise to|result(?:s|ed|ing)? in|"
            r"bring(?:s|ing)? about|brought about|caus(?:e|es|ed|ing)|"
            r"contribut(?:e|es|ed|ing) to|drive|drives|driving|"
            r"produc(?:e|es|ed|ing)|trigger(?:s|ed|ing)?|"
            r"(?:(?:to|can|could|may|might|must|should|will|would|shall)\s+generate|generates|generated|generating)|(?:(?:to|can|could|may|might|must|should|will|would|shall)\s+create|creates|created|creating)|"
            r"exacerbat(?:e|es|ed|ing)|worsen(?:s|ed|ing)?|"
            r"intensif(?:y|ies|ied|ying)|accelerat(?:e|es|ed|ing)|"
            r"increas(?:e|es|ed|ing)|rais(?:e|es|ed|ing)|"
            r"amplif(?:y|ies|ied|ying)|heighten(?:s|ed|ing)?"
        ),
        "causes_or_increases",
    ),
    (
        cause_first_pattern(
            r"reduc(?:e|es|ed|ing)|prevent(?:s|ed|ing)?|"
            r"limit(?:s|ed|ing)?|mitigat(?:e|es|ed|ing)|"
            r"decreas(?:e|es|ed|ing)|lower(?:s|ed|ing)|"
            r"minimi(?:s|z)(?:e|es|ed|ing)|alleviat(?:e|es|ed|ing)|"
            r"avoid(?:s|ed|ing)?|curb(?:s|ed|ing)?|constrain(?:s|ed|ing)?|"
            r"counteract(?:s|ed|ing)?|protect(?:s|ed|ing)? against|"
            r"safeguard(?:s|ed|ing)? against"
        ),
        "reduces_or_prevents",
    ),
    (
        cause_first_pattern(
            r"enabl(?:e|es|ed|ing)|"
            r"(?:(?:to|can|could|may|might|must|should|will|would|shall)\s+support(?:\s+and\s+assist)?|supports|supported|supporting)|"
            r"facilitat(?:e|es|ed|ing)|"
            r"allow(?:s|ed|ing)?(?:\s+for)?|mak(?:e|es|ing) possible|"
            r"made possible|(?:(?:to|can|could|may|might|must|should|will|would|shall)\s+help(?:\s+to|\s+with|\s+in)?|helps(?:\s+to|\s+with|\s+in)?|helped(?:\s+to|\s+with|\s+in)?|helping(?:\s+to|\s+with|\s+in)?)|"
            r"foster(?:s|ed|ing)?|encourag(?:e|es|ed|ing)|"
            r"promot(?:e|es|ed|ing)|empower(?:s|ed|ing)?|"
            r"assist(?:s|ed|ing)?(?:\s+with|\s+in)?"
        ),
        "enables_or_supports",
    ),
    (
        cause_first_pattern(
            r"requir(?:e|es|ed|ing)|depend(?:s|ed|ing)? on|"
            r"rely|relies|relied|relying on|need(?:s|ed|ing)?|"
            r"necessitat(?:e|es|ed|ing)|presuppos(?:e|es|ed|ing)|"
            r"call(?:s|ed|ing)? for|is contingent on|is conditional on"
        ),
        "requires_or_depends_on",
    ),
    (
        cause_first_pattern(
            r"creat(?:e|es|ed|ing) a risk of|"
            r"increas(?:e|es|ed|ing) the risk of|"
            r"rais(?:e|es|ed|ing) the risk of|"
            r"pos(?:e|es|ed|ing) a risk to|risk(?:s|ed|ing)|"
            r"threaten(?:s|ed|ing)?|undermin(?:e|es|ed|ing)|"
            r"harm(?:s|ed|ing)?|jeopardi(?:s|z)(?:e|es|ed|ing)|"
            r"endanger(?:s|ed|ing)?|expos(?:e|es|ed|ing) .+? to|"
            r"compromis(?:e|es|ed|ing)|weaken(?:s|ed|ing)?"
        ),
        "risks_or_threatens",
    ),
    (
        cause_first_pattern(
            r"is expected to improve|is intended to improve|"
            r"(?:(?:to|can|could|may|might|must|should|will|would|shall)\s+improve|improves|improved|improving)|(?:(?:to|can|could|may|might|must|should|will|would|shall)\s+enhance|enhances|enhanced|enhancing)|"
            r"strengthen(?:s|ed|ing)?|advanc(?:e|es|ed|ing)|"
            r"boost(?:s|ed|ing)?|optimi(?:s|z)(?:e|es|ed|ing)|"
            r"rais(?:e|es|ed|ing) the quality of"
        ),
        "expected_to_improve",
    ),
    (
        cause_first_pattern(
            r"peut conduire à|pourrait conduire à|est susceptible de conduire à|"
            r"conduit à|conduit au|conduit aux|donne lieu à|"
            r"entraîn(?:e|es|é|ée|és|ées|ant)|caus(?:e|es|é|ée|és|ées|ant)|"
            r"provoqu(?:e|es|é|ée|és|ées|ant)|"
            r"engendr(?:e|es|é|ée|és|ées|ant)|contribu(?:e|es|é|ée|és|ées|ant) à|"
            r"génèr(?:e|es|é|ée|és|ées|ant)|déclench(?:e|es|é|ée|és|ées|ant)|"
            r"accentu(?:e|es|é|ée|és|ées|ant)|aggrav(?:e|es|é|ée|és|ées|ant)|"
            r"accroît|accroissent|augment(?:e|es|é|ée|és|ées|ant)|"
            r"amplifi(?:e|es|é|ée|és|ées|ant)"
        ),
        "causes_or_increases",
    ),
    (
        cause_first_pattern(
            r"rédui(?:t|ts|te|tes|re|sent|sant)|"
            r"prévien(?:t|nent)|limit(?:e|es|é|ée|és|ées|ant)|"
            r"atténu(?:e|es|é|ée|és|ées|ant)|diminu(?:e|es|é|ée|és|ées|ant)|"
            r"abaiss(?:e|es|é|ée|és|ées|ant)|"
            r"minimis(?:e|es|é|ée|és|ées|ant)|évit(?:e|es|é|ée|és|ées|ant)|"
            r"frein(?:e|es|é|ée|és|ées|ant)|contraint|contraignent|"
            r"protèg(?:e|es|é|ée|és|ées|ant) contre|lutt(?:e|es|é|ée|és|ées|ant) contre"
        ),
        "reduces_or_prevents",
    ),
    (
        cause_first_pattern(
            r"rend possible|rendent possible|permet(?:s|tent)? de|"
            r"permet(?:s|tent)?|soutien(?:t|nent)|facilit(?:e|es|é|ée|és|ées|ant)|"
            r"favoris(?:e|es|é|ée|és|ées|ant)|encourag(?:e|es|é|ée|és|ées|ant)|"
            r"aid(?:e|es|é|ée|és|ées|ant) à|donne les moyens de"
        ),
        "enables_or_supports",
    ),
    (
        cause_first_pattern(
            r"nécessit(?:e|es|é|ée|és|ées|ant)|dépend(?:s|ent)? de|"
            r"repos(?:e|es|é|ée|és|ées|ant) sur|exig(?:e|es|é|ée|és|ées|ant)|"
            r"requiert|requièrent|suppos(?:e|es|é|ée|és|ées|ant)|"
            r"appell(?:e|es|é|ée|és|ées|ant)|est tributaire de|"
            r"est conditionné par|est conditionnée par"
        ),
        "requires_or_depends_on",
    ),
    (
        cause_first_pattern(
            r"crée un risque de|créent un risque de|accroît le risque de|"
            r"augmente le risque de|présente un risque pour|"
            r"menac(?:e|es|é|ée|és|ées|ant)|risque de|"
            r"compromet|compromettent|nui(?:t|sent) à|met en danger|"
            r"mettent en danger|fragilis(?:e|es|é|ée|és|ées|ant)|"
            r"expos(?:e|es|é|ée|és|ées|ant) .+? à"
        ),
        "risks_or_threatens",
    ),
    (
        cause_first_pattern(
            r"devrait améliorer|vise à améliorer|"
            r"amélior(?:e|es|é|ée|és|ées|ant)|"
            r"renforc(?:e|es|é|ée|és|ées|ant)|"
            r"optimis(?:e|es|é|ée|és|ées|ant)|accroît la qualité de"
        ),
        "expected_to_improve",
    ),
]

REVERSED_PATTERNS = [
    (
        reversed_pattern(
            r"because of|because|due to|owing to|as a result of|"
            r"resulting from|caused by|driven by|triggered by|produced by|"
            r"generated by|enabled by|facilitated by|made possible by"
        ),
        "causes_or_increases",
    ),
    (
        reversed_pattern(
            r"depends? on|relies? on|is contingent on|is conditional on"
        ),
        "requires_or_depends_on",
    ),
    (
        reversed_pattern(
            r"parce que|en raison de|à cause de|du fait de|résultant de|"
            r"causé par|causée par|provoqué par|provoquée par|"
            r"entraîné par|entraînée par|généré par|générée par|"
            r"rendu possible par|rendue possible par"
        ),
        "causes_or_increases",
    ),
    (
        reversed_pattern(
            r"dépend de|repose sur|est tributaire de|"
            r"est conditionné par|est conditionnée par"
        ),
        "requires_or_depends_on",
    ),
]

PURPOSE_PATTERNS = [
    (
        cause_first_pattern(
            r"in order to|so as to|with the aim of|for the purpose of|"
            r"with a view to|designed to|intended to|"
            r"to help|to enable|to ensure|to prevent|to reduce"
        ),
        "enables_or_supports",
    ),
    (
        cause_first_pattern(
            r"afin de|dans le but de|en vue de|de manière à|de façon à|"
            r"destiné à|destinée à|conçu pour|conçue pour|visant à|"
            r"pour permettre de|pour aider à|pour assurer|pour éviter|"
            r"pour réduire"
        ),
        "enables_or_supports",
    ),
]


def split_sentences(text: str) -> list[str]:
    cleaned = html.unescape(str(text))
    cleaned = re.sub(r"\s+[•●▪◦]\s+", ". ", cleaned)
    cleaned = re.sub(
        r"\s+-\s+(?=[A-ZÀ-ÖØ-Ý][A-Za-zÀ-ÖØ-öø-ÿ ]{2,}:)",
        ". ",
        cleaned,
    )
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    if not cleaned:
        return []
    return [
        sentence.strip(" -•\t")
        for sentence in re.split(
            r"(?<=[.!?])\s+(?=[A-ZÀ-ÖØ-Ý0-9])",
            cleaned,
        )
        if sentence.strip()
    ]


def clean_side(value: str) -> str:
    cleaned = normalize_for_noise_check(value)
    cleaned = re.sub(
        r"^(?:chapter|section|article|figure|table|appendix|annex|"
        r"chapitre|partie|article|figure|tableau|annexe)\s+[A-Za-z0-9.:-]+\s*",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(
        r"^(?:and|or|et|ou)\s+",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(
        r"\s+(?:to|de|à|pour)$",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(r"\s+", " ", cleaned).strip(" ,;:-")
    return cleaned[:500]


def is_low_information_side(value: str) -> bool:
    """Reject sides dominated by source artifacts or fragmentary tokens."""
    tokens = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9]+", str(value).lower())
    if len(tokens) < MIN_SIDE_WORDS:
        return True
    if len(tokens) > MAX_SIDE_WORDS:
        return True
    if content_token_ratio(value) < 0.45:
        return True
    artifact_hits = sum(token in CAUSAL_ARTIFACT_TOKENS for token in tokens)
    return artifact_hits >= 3


AMBIGUOUS_BASE_CUES = {
    "need", "risk", "limit", "increase", "decrease",
    "allow", "promote", "facilitate",
    "strengthen", "reduce", "prevent", "require",
}

NOUN_PREDECESSORS = {
    "a", "an", "the", "this", "that", "these", "those", "of", "for",
    "with", "without", "under", "over", "additional", "administrative",
    "continued", "digital", "educational", "existing", "financial",
    "human", "institutional", "ongoing", "practical", "professional",
    "social", "technical", "alongside", "through", "via",
    "including", "concerning", "regarding", "around",
}

SUPPORT_NOUN_FOLLOWERS = {
    "staff", "worker", "workers", "service", "services", "team", "teams",
    "function", "functions", "material", "materials", "resource",
    "resources", "network", "networks", "group", "groups", "structure",
    "structures", "mechanism", "mechanisms", "work", "personnel",
}


def is_plausible_cue_usage(sentence: str, match: re.Match) -> bool:
    """Reject common noun/adjective readings of otherwise causal cue words."""
    cue = re.sub(r"\\s+", " ", match.group("cue").lower()).strip()
    cue_start, cue_end = match.span("cue")
    before_tokens = re.findall(
        r"[A-Za-zÀ-ÖØ-öø-ÿ]+",
        sentence[:cue_start].lower(),
    )
    after_tokens = re.findall(
        r"[A-Za-zÀ-ÖØ-öø-ÿ]+",
        sentence[cue_end:].lower(),
    )
    previous_word = before_tokens[-1] if before_tokens else ""
    next_word = after_tokens[0] if after_tokens else ""

    if cue in AMBIGUOUS_BASE_CUES and previous_word in NOUN_PREDECESSORS:
        return False

    if cue == "support":
        if previous_word in NOUN_PREDECESSORS:
            return False
        if previous_word in {
            "provide", "provides", "provided", "providing",
            "obtain", "obtains", "obtained", "obtaining",
            "receive", "receives", "received", "receiving",
            "funding", "scientific",
        }:
            return False
        if next_word in SUPPORT_NOUN_FOLLOWERS | {
            "for", "of", "from", "to", "and", "or",
        }:
            return False

    if cue == "help":
        if previous_word in NOUN_PREDECESSORS:
            return False
        if next_word in {"from", "of", "for", "and", "or"}:
            return False

    if cue == "need":
        if previous_word in {"the", "a", "an", "this", "that", "identified"}:
            return False
        if next_word in {"for", "of"}:
            return False

    if cue == "risk":
        return False

    if cue in {"increase", "decrease", "limit"}:
        if previous_word in {"the", "a", "an", "this", "that", "percentage"}:
            return False

    if cue == "risks":
        if previous_word in {
            "the", "a", "an", "other", "potential", "greatest",
            "significant", "various", "several", "these", "those",
            "related", "associated",
        }:
            return False
        if next_word in {
            "is", "are", "was", "were", "that", "to", "across",
            "related", "associated", "identified", "include", "including",
        }:
            return False

    if cue in {
        "improved", "enhanced", "strengthened", "increased",
        "reduced", "limited", "generated", "created",
    }:
        if previous_word in {
            "a", "an", "the", "and", "or", "new", "better", "more",
            "less", "with", "for", "of", "is", "are", "was", "were",
            "be", "been", "being", "remains", "remained",
        }:
            return False

    if cue.endswith("ing") and previous_word in {
        "in", "for", "of", "by", "through", "while",
        "after", "before", "during",
    }:
        return False

    if cue == "need" and next_word in {
        "paper", "report", "document", "study", "article",
    }:
        return False

    if cue == "required" and (
        next_word == "by"
        or previous_word in {"as", "is", "are", "was", "were"}
    ):
        return False

    # A cue followed only by punctuation, a conjunction, or a stranded
    # preposition does not yield a meaningful effect phrase.
    if next_word in {
        "and", "or", "et", "ou", "for", "of", "from", "with",
    }:
        return False

    return True


def extract_claims(frame: pd.DataFrame, corpus_type: str) -> pd.DataFrame:
    rows = []
    counter = 0
    for _, document in frame.iterrows():
        for sentence_index, sentence in enumerate(split_sentences(document["chunk_text"])):
            lowered = sentence.lower()
            words = sentence.split()
            if (
                len(words) > MAX_SENTENCE_WORDS
                or is_source_residue(sentence)
            ):
                continue
            result = None
            for pattern, relation_type in [*REVERSED_PATTERNS, *PURPOSE_PATTERNS, *CAUSE_FIRST_PATTERNS]:
                match = re.match(pattern, sentence, flags=re.IGNORECASE)
                if not match:
                    continue
                if not is_plausible_cue_usage(sentence, match):
                    continue
                cause = clean_side(match.group("cause"))
                effect = clean_side(match.group("effect"))
                if not is_low_information_side(cause) and not is_low_information_side(effect):
                    result = (cause, relation_type, effect, match.group("cue"))
                    break
            if result is None:
                continue
            cause, relation_type, effect, cue = result
            if (
                contains_contact_or_link(cause)
                or contains_contact_or_link(effect)
                or "|" in sentence
                or "|" in cause
                or "|" in effect
            ):
                continue

            claim_signature = re.sub(
                r"[^a-zà-öø-ÿ0-9]+",
                " ",
                f"{cause} {relation_type} {effect}".lower(),
            ).strip()

            cleaned_sentence = normalize_for_noise_check(sentence)
            rows.append({
                "claim_id": f"{corpus_type}_claim_{counter:05d}",
                "corpus_type": corpus_type,
                "doc_id": document["doc_id"],
                "chunk_id": document["chunk_id"],
                "sentence_id": document["sentence_id"],
                "chunk_index": int(document["chunk_index"]),
                "filename": document["filename"],
                "source_file": document["source_file"],
                "country": document["analysis_country"],
                "heading_context": document.get("heading_context", ""),
                "source_text": cleaned_sentence,
                "context_window": document.get("context_window", ""),
                "cause": cause,
                "relation_type": relation_type,
                "relation_cue": cue,
                "effect": effect,
                "structured_claim": (
                    f"{cause} [{relation_type}] {effect}"
                ),
                "canonical_claim": cleaned_sentence,
                "extraction_method": "sentence_level_first_valid_pattern",
                "review_status": "",
                "review_notes": "",
                "claim_signature": claim_signature,
            })
            counter += 1
    result = pd.DataFrame(rows)
    if result.empty:
        return result
    # Adjacent chunks may repeat the same sentence. Keep one occurrence per
    # document while preserving identical claims from genuinely different sources.
    result = result.drop_duplicates(
        subset=["corpus_type", "doc_id", "claim_signature"]
    ).reset_index(drop=True)
    return result.drop(columns=["claim_signature"])


policy_claims = extract_claims(policy_chunks, "policy")
sentiment_claims = extract_claims(sentiment_chunks, "sentiment")
synthetic_claims = extract_claims(synthetic_chunks, "synthetic_sentiment")
if policy_claims.empty or sentiment_claims.empty:
    raise ValueError("No empirical claims were extracted.")

pd.concat([policy_claims, sentiment_claims], ignore_index=True).to_csv(
    OUTPUT_DIR / "causal_claim_extraction_audit.csv", index=False
)
print("Policy claims:", len(policy_claims))
print("Sentiment claims:", len(sentiment_claims))
print("Synthetic claims:", len(synthetic_claims))
print(pd.concat([policy_claims, sentiment_claims])["relation_type"].value_counts().to_string())


Policy claims: 3998
Sentiment claims: 836
Synthetic claims: 782
relation_type
enables_or_supports       2123
causes_or_increases       1332
requires_or_depends_on     700
expected_to_improve        300
reduces_or_prevents        272
risks_or_threatens         107


## Descriptive claim inventory
Inspect the source balance and relation families before comparing embeddings.


In [4]:
# Summarize extracted cause–relation–effect statements before matching.
for label, frame in (("policy", policy_claims), ("commentary", sentiment_claims),
                     ("synthetic_probe", synthetic_claims)):
    frame.to_csv(OUTPUT_DIR / f"{label}_causal_claims.csv", index=False)
    print(label, "claims", len(frame))
print("Relation families across empirical claims:")
print(pd.concat([policy_claims, sentiment_claims])["relation_type"].value_counts().to_string())


policy claims 3998
commentary claims 836
synthetic_probe claims 782
Relation families across empirical claims:
relation_type
enables_or_supports       2123
causes_or_increases       1332
requires_or_depends_on     700
expected_to_improve        300
reduces_or_prevents        272
risks_or_threatens         107


## Global directed coverage
Use top-3 nearest neighbors, the original threshold calibration, and 50 balanced resamples.


In [5]:
# --- Analysis stage ---------------------------------------------------------

TOP_K = 3
BALANCED_RESAMPLES = 50
LOW_COVERAGE_QUANTILE = 0.10
THRESHOLD_AUDIT_PER_BAND = 20
MANUAL_THRESHOLD_FILE = OUTPUT_DIR / "semantic_match_threshold_manual.csv"

all_texts = (
    policy_claims["canonical_claim"].tolist()
    + sentiment_claims["canonical_claim"].tolist()
)
all_embeddings, embedding_backend = fit_text_embeddings(all_texts)
n_policy = len(policy_claims)
policy_embeddings = all_embeddings[:n_policy]
sentiment_embeddings = all_embeddings[n_policy:]


def normalized_similarity_matrix(left, right) -> np.ndarray:
    """Multiply already-normalized embeddings without a second normalization pass."""
    product = left @ right.T
    if hasattr(product, "toarray"):
        product = product.toarray()
    return np.asarray(product, dtype=np.float32)


def build_matches(
    source: pd.DataFrame,
    target: pd.DataFrame,
    source_embeddings: np.ndarray,
    target_embeddings: np.ndarray,
    direction: str,
) -> pd.DataFrame:
    """Return traceable nearest-neighbour matches without applying a threshold."""
    if len(source) == 0 or len(target) == 0:
        return pd.DataFrame()

    matrix = normalized_similarity_matrix(source_embeddings, target_embeddings)
    k = min(TOP_K, matrix.shape[1])
    order = np.argsort(matrix, axis=1)[:, ::-1][:, :k]
    scores = np.take_along_axis(matrix, order, axis=1)

    rows = []
    source_reset = source.reset_index(drop=True)
    target_reset = target.reset_index(drop=True)
    for index, source_row in source_reset.iterrows():
        best = target_reset.iloc[int(order[index, 0])]
        mean_similarity = float(scores[index].mean())
        rows.append({
            "direction": direction,
            "source_claim_id": source_row["claim_id"],
            "source_chunk_id": source_row["chunk_id"],
            "source_doc_id": source_row["doc_id"],
            "source_country": source_row["country"],
            "source_relation_type": source_row["relation_type"],
            "source_claim": source_row["canonical_claim"],
            "source_excerpt": source_row["source_text"],
            "matched_claim_id": best["claim_id"],
            "matched_chunk_id": best["chunk_id"],
            "matched_claim": best["canonical_claim"],
            "matched_excerpt": best["source_text"],
            "top1_similarity": float(scores[index, 0]),
            "topk_mean_similarity": mean_similarity,
            "coverage_deficit": float(1.0 - mean_similarity),
            "embedding_method": embedding_backend["method"],
        })
    return pd.DataFrame(rows)


sentiment_to_policy = build_matches(
    sentiment_claims,
    policy_claims,
    sentiment_embeddings,
    policy_embeddings,
    "sentiment_to_policy",
)
policy_to_sentiment = build_matches(
    policy_claims,
    sentiment_claims,
    policy_embeddings,
    sentiment_embeddings,
    "policy_to_sentiment",
)
global_matches = pd.concat(
    [sentiment_to_policy, policy_to_sentiment],
    ignore_index=True,
)


def parse_manual_label(value) -> bool:
    """Convert common manual-review values to a Boolean match label."""
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    if text in {"1", "true", "yes", "match", "matched"}:
        return True
    if text in {"0", "false", "no", "nonmatch", "non-match", "unmatched"}:
        return False
    raise ValueError(f"Unrecognised manual semantic-match label: {value!r}")


def calibrate_threshold(
    matches: pd.DataFrame,
    manual_path: Path,
) -> tuple[float, str]:
    """Use reviewed pairs when available; otherwise use a lower-tail diagnostic."""
    if manual_path.exists():
        reviewed = pd.read_csv(manual_path)
        score_column = (
            "topk_mean_similarity"
            if "topk_mean_similarity" in reviewed.columns
            else "similarity"
        )
        label_column = (
            "is_semantic_match"
            if "is_semantic_match" in reviewed.columns
            else "review_is_match"
        )
        required = {score_column, label_column}
        if not required.issubset(reviewed.columns):
            raise ValueError(
                "Manual threshold file must contain a similarity score and "
                "is_semantic_match/review_is_match."
            )

        scores = pd.to_numeric(
            reviewed[score_column],
            errors="coerce",
        )
        labels = reviewed[label_column].map(parse_manual_label)
        valid = scores.notna()
        scores = scores[valid].to_numpy(dtype=float)
        labels = labels[valid].to_numpy(dtype=bool)
        if len(scores) < 10 or labels.sum() == 0 or (~labels).sum() == 0:
            raise ValueError(
                "Manual threshold calibration needs at least 10 reviewed pairs "
                "containing both matches and non-matches."
            )

        candidates = np.unique(scores)
        best = None
        for threshold in candidates:
            predicted = scores >= threshold
            tp = int(np.sum(predicted & labels))
            fp = int(np.sum(predicted & ~labels))
            fn = int(np.sum(~predicted & labels))
            precision = tp / (tp + fp) if tp + fp else 0.0
            recall = tp / (tp + fn) if tp + fn else 0.0
            f1 = (
                2 * precision * recall / (precision + recall)
                if precision + recall
                else 0.0
            )
            candidate = (f1, precision, recall, float(threshold))
            if best is None or candidate > best:
                best = candidate

        return best[3], "manual_review_f1_calibration"

    threshold = float(
        matches["topk_mean_similarity"].quantile(LOW_COVERAGE_QUANTILE)
    )
    return threshold, (
        f"pooled_empirical_{int(LOW_COVERAGE_QUANTILE * 100)}th_"
        "percentile_provisional"
    )


coverage_threshold, threshold_method = calibrate_threshold(
    global_matches,
    MANUAL_THRESHOLD_FILE,
)
global_matches["below_calibrated_threshold"] = (
    global_matches["topk_mean_similarity"] < coverage_threshold
)
global_matches["coverage_threshold"] = coverage_threshold
global_matches["threshold_method"] = threshold_method
global_matches.to_csv(
    OUTPUT_DIR / "global_semantic_claim_matches.csv",
    index=False,
)

# Export a stratified review sheet for later threshold calibration.
audit_source = global_matches.copy()
audit_source["similarity_band"] = pd.qcut(
    audit_source["topk_mean_similarity"],
    q=[0.0, 0.10, 0.50, 0.90, 1.0],
    labels=["lowest_10pct", "lower_middle", "upper_middle", "highest_10pct"],
    duplicates="drop",
)
audit_parts = []
for _, group in audit_source.groupby(
    ["direction", "similarity_band"],
    observed=True,
):
    audit_parts.append(
        group.sample(
            n=min(THRESHOLD_AUDIT_PER_BAND, len(group)),
            random_state=RANDOM_STATE,
        )
    )
threshold_audit = pd.concat(audit_parts, ignore_index=True)
threshold_audit["review_is_match"] = ""
threshold_audit["review_notes"] = ""
threshold_audit.to_csv(
    OUTPUT_DIR / "semantic_match_threshold_audit.csv",
    index=False,
)

global_summary = global_matches.groupby(
    "direction",
    as_index=False,
).agg(
    claims=("source_claim_id", "count"),
    mean_top1_similarity=("top1_similarity", "mean"),
    mean_topk_similarity=("topk_mean_similarity", "mean"),
    mean_coverage_deficit=("coverage_deficit", "mean"),
    low_coverage_share=("below_calibrated_threshold", "mean"),
)
global_summary["coverage_threshold"] = coverage_threshold
global_summary["threshold_method"] = threshold_method
global_summary.to_csv(
    OUTPUT_DIR / "global_semantic_coverage_summary.csv",
    index=False,
)


# Bootstrap balanced claim subsets to check directional sensitivity to sample composition.
BALANCED_SAMPLE_SIZE = min(len(policy_claims), len(sentiment_claims))
policy_vs_commentary = normalized_similarity_matrix(policy_embeddings, sentiment_embeddings)
rng = np.random.default_rng(RANDOM_STATE)
bootstrap_rows = []
for repeat in range(BALANCED_RESAMPLES):
    left = rng.choice(len(policy_claims), size=BALANCED_SAMPLE_SIZE, replace=True)
    right = rng.choice(len(sentiment_claims), size=BALANCED_SAMPLE_SIZE, replace=True)
    sampled_matrix = policy_vs_commentary[np.ix_(left, right)]
    for direction, values in (
        ("policy_to_sentiment", sampled_matrix),
        ("sentiment_to_policy", sampled_matrix.T),
    ):
        k = min(TOP_K, values.shape[1])
        topk = np.sort(values, axis=1)[:, -k:].mean(axis=1)
        bootstrap_rows.append({"repeat": repeat, "direction": direction,
                               "mean_coverage_deficit": float((1 - topk).mean()),
                               "sampled_claims": len(topk)})
balanced_runs = pd.DataFrame(bootstrap_rows)
balanced_runs.to_csv(OUTPUT_DIR / "global_balanced_coverage_resamples.csv", index=False)
balanced_summary = balanced_runs.groupby("direction", as_index=False).agg(
    mean_deficit=("mean_coverage_deficit", "mean"),
    std_deficit=("mean_coverage_deficit", "std"),
    resamples=("repeat", "count"),
)
balanced_summary.to_csv(OUTPUT_DIR / "global_balanced_coverage_summary.csv", index=False)
print(global_summary.to_string(index=False))
print(balanced_summary.to_string(index=False))


Multilingual Sentence-BERT unavailable; using multilingual word+character TF-IDF plus LSA: OSError
          direction  claims  mean_top1_similarity  mean_topk_similarity  mean_coverage_deficit  low_coverage_share  coverage_threshold                             threshold_method
policy_to_sentiment    3998              0.674841              0.644474               0.355526            0.121061            0.569776 pooled_empirical_10th_percentile_provisional
sentiment_to_policy     836              0.726494              0.700407               0.299593            0.000000            0.569776 pooled_empirical_10th_percentile_provisional
          direction  mean_deficit  std_deficit  resamples
policy_to_sentiment      0.364209     0.003142         50
sentiment_to_policy      0.362703     0.003674         50


## Evidence review sample
Rank potential mismatches by their recorded coverage deficit, retaining the source and matching claim.


In [6]:
# Keep supporting excerpts for the largest observed directional gaps.
global_evidence = global_matches.sort_values(
    ["coverage_deficit", "top1_similarity"], ascending=[False, True]
).groupby("direction", group_keys=False).head(50).reset_index(drop=True)
global_evidence.to_csv(OUTPUT_DIR / "high_gap_evidence_pairs.csv", index=False)
print(global_evidence[["direction", "source_claim_id", "matched_claim_id", "top1_similarity"]].head(10))


             direction     source_claim_id       matched_claim_id  top1_similarity
0  policy_to_sentiment  policy_claim_01493  sentiment_claim_00786         0.488702
1  policy_to_sentiment  policy_claim_01138  sentiment_claim_00646         0.495812
2  policy_to_sentiment  policy_claim_02771  sentiment_claim_00557         0.531457
3  policy_to_sentiment  policy_claim_01370  sentiment_claim_00089         0.523182
4  policy_to_sentiment  policy_claim_00373  sentiment_claim_00634         0.510330
5  policy_to_sentiment  policy_claim_02122  sentiment_claim_00571         0.529114
6  policy_to_sentiment  policy_claim_00998  sentiment_claim_00497         0.516561
7  policy_to_sentiment  policy_claim_01174  sentiment_claim_00827         0.542819
8  policy_to_sentiment  policy_claim_02642  sentiment_claim_00555         0.519216
9  policy_to_sentiment  policy_claim_02648  sentiment_claim_00212         0.502599


## Synthetic perturbation
Keep synthetic sentences separate from empirical findings.


In [7]:
# Treat the synthetic supplement strictly as a sensitivity probe.
if not synthetic_claims.empty:
    synthetic_embeddings = transform_text_embeddings(synthetic_claims["canonical_claim"].tolist(), embedding_backend)
    synthetic_matches = build_matches(
        synthetic_claims, policy_claims, synthetic_embeddings, policy_embeddings,
        "synthetic_to_policy",
    )
    synthetic_matches.to_csv(OUTPUT_DIR / "synthetic_probe_claim_matches.csv", index=False)
    synthetic_sensitivity = pd.DataFrame([{
        "empirical_commentary_mean_deficit": float(sentiment_to_policy["coverage_deficit"].mean()),
        "synthetic_probe_mean_deficit": float(synthetic_matches["coverage_deficit"].mean()),
        "difference": float(synthetic_matches["coverage_deficit"].mean() - sentiment_to_policy["coverage_deficit"].mean()),
        "empirical_claims": len(sentiment_claims), "synthetic_claims": len(synthetic_claims),
    }])
else:
    synthetic_sensitivity = pd.DataFrame()
if not synthetic_sensitivity.empty:
    synthetic_sensitivity.to_csv(OUTPUT_DIR / "synthetic_coverage_sensitivity.csv", index=False)
    print(synthetic_sensitivity.to_string(index=False))
else:
    print("No synthetic claims available for the optional comparison.")


 empirical_commentary_mean_deficit  synthetic_probe_mean_deficit  difference  empirical_claims  synthetic_claims
                          0.299593                      0.299774     0.00018               836               782


## Static visualization
Show the similarity distribution and mean gap by direction.


In [8]:
# Render global summaries with Matplotlib, avoiding WebGL requirements.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for direction, values in global_matches.groupby("direction"):
    axes[0].hist(values["topk_mean_similarity"], bins=24, alpha=0.5, label=direction)
axes[0].set(title="Cross-corpus claim similarity", xlabel="Mean top-k cosine similarity", ylabel="Claims")
axes[0].legend()
axes[1].bar(global_summary["direction"], global_summary["mean_coverage_deficit"])
axes[1].set(title="Global directional gap", ylabel="Mean coverage deficit")
axes[1].tick_params(axis="x", rotation=18)
fig.tight_layout()
save_verified_png(fig, IMG_DIR / "global_claim_coverage.png")


Saved verified PNG: /home/mouelhis/Private/github/new_dsac/cnlp/causal_semantics/figures/semantic_coverage/global_claim_coverage.png mode=RGB size=(1428, 468) bytes=45,542


## Run summary and limitations
Coverage is about *expressed claims*; it does not establish causal effects.


In [9]:
# Report the implemented scope, data sizes, and interpretation boundaries.
run_summary = pd.DataFrame([{
    "method": "global_causal_claim_alignment", "scope": "pooled_global",
    "topic_dependencies": 0, "embedding_method": embedding_backend["method"],
    "input_file": CHUNKS_ALL.name, "sentence_cache": CLEAN_SENTENCE_INVENTORY.name,
    "policy_claims": len(policy_claims), "commentary_claims": len(sentiment_claims),
    "synthetic_probe_claims": len(synthetic_claims), "top_k": TOP_K,
    "balanced_resamples": BALANCED_RESAMPLES,
    "threshold": coverage_threshold, "threshold_method": threshold_method,
    "interpretation": "Textual overlap is not evidence of real-world causal effects",
}])
run_summary.to_csv(OUTPUT_DIR / "run_summary.csv", index=False)
print(run_summary.to_string(index=False))


                       method         scope  topic_dependencies                         embedding_method                   input_file              sentence_cache  policy_claims  commentary_claims  synthetic_probe_claims  top_k  balanced_resamples  threshold                             threshold_method                                               interpretation
global_causal_claim_alignment pooled_global                   0 multilingual_word_char_lsa:60_components policy_feedback_segments.csv sentence_evidence_cache.csv           3998                836                     782      3                  50   0.569776 pooled_empirical_10th_percentile_provisional Textual overlap is not evidence of real-world causal effects
